# Run and visualize one physical design

No AI model or MATLAB is needed. Install `.[analysis,notebooks]` and select that Python environment as the notebook kernel. This example starts supported and simulates 0.2 seconds. It writes a YAML copy so the original example stays available.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess
ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), "Open this notebook from the repository or tutorials directory."
os.chdir(ROOT)
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ.setdefault("MPLCONFIGDIR", "/tmp/walking-robot-mpl")
print("Repository:", ROOT)
print("Python:", sys.executable)


In [ ]:
import yaml
from IPython.display import Image, display
configuration = yaml.safe_load(Path("examples/designs/matlab_supported.yaml").read_text())
configuration["simulation"]["duration_s"] = 0.2
configuration["visualization"].update(formats=["html"], max_frames=150)
# Change thickness in micrometers, for example 1.0 to 2.0:
configuration["design"]["adhesive_thickness_um"] = 1.0
OUTPUT = Path("outputs/tutorial_one_design")
OUTPUT.mkdir(parents=True, exist_ok=True)
DESIGN = OUTPUT / "input_design.yaml"
DESIGN.write_text(yaml.safe_dump(configuration, sort_keys=False))
print(DESIGN.read_text())


In [ ]:
subprocess.run([sys.executable, "examples/run_design.py", "--design", str(DESIGN), "--output", str(OUTPUT)], check=True)
report = json.loads((OUTPUT / "validation.json").read_text())
print("Physical screening passed:", report["passed"])
print("Violations:", report["violations"])
print("Mean COM speed in the last half (m/s):", report["metrics"].get("late_mean_com_speed_m_s"))
display(Image(filename=str(OUTPUT / "diagnostics.png")))


Open `outputs/tutorial_one_design/robot.html` for the animation. Movie speed and frame count change only the visualization. Physical checks use every integration step. A passing screen means provisional limits were met; it is not a measurement of hardware performance.

Try another thickness and a new output directory. See `docs/design_parameters.md` for units and `02_compare_designs.ipynb` to automate multiple cases.
